# One GRPO Step on One SWE-bench Task

**Goal: conceptual clarity, not training.** Nothing here converges. By the end
you will have watched a single gradient update travel the entire pipeline, with
every intermediate value printed.

The five stages, and nothing else:

1. Load **one** instance from SWE-bench Verified
2. A minimal mini-swe-agent-shaped harness produces rollouts
3. Every bash call is **mocked** — no Docker, no repo clone, no execution
4. The final patch gets a reward
5. A group of rewards becomes **one** GRPO step

### The one thing to hold onto

You can mock the filesystem. You can mock `grep`, `cat`, and `pytest`. You
cannot mock the reward — a real reward means really running the tests in a real
environment. §4 is where the pretending stops working, and §7 keeps the ledger
honest.

Runs on a T4 in a few minutes. Also runs on CPU if you're patient.

In [18]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" "peft>=0.12" torch

import torch, re, json, difflib, random
import numpy as np
import pandas as pd
from IPython.display import display

for _opt in ["display.max_colwidth", "display.max_rows", "display.max_columns", "display.width"]:
    pd.set_option(_opt, None)

random.seed(0); torch.manual_seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

device: cuda


---
# 1. Load one instance

500 human-validated tasks. We take one and read it field by field — a SWE-bench
task is five artifacts, and only the first is the agent's input.

In [19]:
from datasets import load_dataset

ds = load_dataset("princeton-nlp/SWE-bench_Verified", split="test")
print(ds)

# A small, single-file task keeps the walkthrough readable.
cands = [i for i, r in enumerate(ds)
         if r["patch"].count("diff --git") == 1 and len(r["patch"]) < 1800]
inst = ds[cands[0]]
fail_to_pass = json.loads(inst["FAIL_TO_PASS"])
pass_to_pass = json.loads(inst["PASS_TO_PASS"])

display(pd.DataFrame(
    [(k, inst.get(k)) for k in ["instance_id", "repo", "base_commit", "version", "difficulty"]],
    columns=["field", "value"],
))

display(pd.DataFrame([
    ("1.1", "problem_statement", "the agent",  "the input: a raw GitHub issue"),
    ("-",   "patch",             "nobody",     "reference solution; unused in this notebook"),
    ("1.2", "test_patch",        "the grader", "adds the tests that define 'fixed'"),
    ("1.3", "FAIL_TO_PASS",      "the grader", "must go red -> green"),
    ("1.4", "PASS_TO_PASS",      "the grader", "must stay green"),
], columns=["section", "field", "who sees it", "job"]))

Dataset({
    features: ['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit', 'difficulty'],
    num_rows: 500
})


,field,value
0,instance_id,astropy__astropy-12907
1,repo,astropy/astropy
2,base_commit,d16bfe05a744909de4b27f5875fe0d4ed41ce607
3,version,4.3
4,difficulty,15 min - 1 hour


,section,field,who sees it,job
0,1.1,problem_statement,the agent,the input: a raw GitHub issue
1,-,patch,nobody,reference solution; unused in this notebook
2,1.2,test_patch,the grader,adds the tests that define 'fixed'
3,1.3,FAIL_TO_PASS,the grader,must go red -> green
4,1.4,PASS_TO_PASS,the grader,must stay green


## 1.1 `problem_statement` — the agent's entire input

A raw GitHub issue. The agent is never told which file to open, or that
`separable.py` exists. That localization step is most of the real difficulty —
and the first thing §3's mock deletes.

In [20]:
print(inst["problem_statement"])

Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))
array([[ True,  True, False, False],
       [ True,  True, False, False],
       [False, False,  True, False],
       [False, False, False,  True]])
```

The output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.

If however, I nest these compound models:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & cm)
array([[ True,  True, False, False],
       [ True,  True, False, 

## 1.2 `test_patch` — the grader's tests (held out, always applied)

The real PR changed two things: `separable.py` (the fix) and `test_separable.py`
(the regression tests). SWE-bench splits that PR — the source half becomes
`inst["patch"]`, which nothing here uses, and the test half becomes `test_patch`
— then pins `base_commit` to just before it.

So **these tests do not exist in the agent's repo.** Grading has to inject them
or nothing can detect the bug. The agent must not see them either: the expected
matrices are the answer.

In [21]:
print(inst["test_patch"])

diff --git a/astropy/modeling/tests/test_separable.py b/astropy/modeling/tests/test_separable.py
--- a/astropy/modeling/tests/test_separable.py
+++ b/astropy/modeling/tests/test_separable.py
@@ -28,6 +28,13 @@
 p1 = models.Polynomial1D(1, name='p1')
 
 
+cm_4d_expected = (np.array([False, False, True, True]),
+                  np.array([[True,  True,  False, False],
+                            [True,  True,  False, False],
+                            [False, False, True,  False],
+                            [False, False, False, True]]))
+
+
 compound_models = {
     'cm1': (map3 & sh1 | rot & sh1 | sh1 & sh2 & sh1,
             (np.array([False, False, True]),
@@ -52,7 +59,17 @@
     'cm7': (map2 | p2 & sh1,
             (np.array([False, True]),
              np.array([[True, False], [False, True]]))
-            )
+            ),
+    'cm8': (rot & (sh1 & sh2), cm_4d_expected),
+    'cm9': (rot & sh1 & sh2, cm_4d_expected),
+    'cm10': ((rot & sh1) & sh2, cm_4d_expected),
+    

## 1.3 `FAIL_TO_PASS` — must go red → green

The half of the reward that says *you solved the issue*.

The node ids look unrelated to the `cm*` keys above because `parametrize`
numbers cases positionally: `compound_models` holds 6 entries at `base_commit`
(`cm1`–`cm5`, `cm7`), and `test_patch` appends 4 more at indices 6–9.

The two that fail are exactly the **right-nested** ones — which is precisely
what the fix repairs, the `cright` branch you'll meet in §2.

In [22]:
for t in fail_to_pass:
    print(t)

new_cases = pd.DataFrame([
    (6, "cm8",  "rot & (sh1 & sh2)",         "right-nested"),
    (7, "cm9",  "rot & sh1 & sh2",           "flat"),
    (8, "cm10", "(rot & sh1) & sh2",         "left-nested"),
    (9, "cm11", "rot & sh1 & (scl1 & scl2)", "right-nested"),
], columns=["index", "case", "model", "grouping"])
new_cases.insert(1, "node_id", "compound_model" + new_cases["index"].astype(str))
new_cases["in_FAIL_TO_PASS"] = [
    any(f"[{n}-" in t for t in fail_to_pass) for n in new_cases["node_id"]
]
display(new_cases)

astropy/modeling/tests/test_separable.py::test_separable[compound_model6-result6]
astropy/modeling/tests/test_separable.py::test_separable[compound_model9-result9]


,index,node_id,case,model,grouping,in_FAIL_TO_PASS
0,6,compound_model6,cm8,rot & (sh1 & sh2),right-nested,True
1,7,compound_model7,cm9,rot & sh1 & sh2,flat,False
2,8,compound_model8,cm10,(rot & sh1) & sh2,left-nested,False
3,9,compound_model9,cm11,rot & sh1 & (scl1 & scl2),right-nested,True


## 1.4 `PASS_TO_PASS` — must stay green

The anti-reward-hacking half: without it, deleting the two failing tests would
score a win. SWE-bench also resets test files to `base_commit` before applying
`test_patch`, so tampering is discarded twice over.

Note that two of the four *new* cases land here, not in `FAIL_TO_PASS`. A new
test is not automatically a target — the lists are built by running the suite at
`base_commit` and sorting by observed outcome.

In [23]:
for t in pass_to_pass:
    print(t)

display(pd.DataFrame([
    ("existing cases, index 0-5",   6, "PASS_TO_PASS"),
    ("new, already green (7, 8)",   2, "PASS_TO_PASS"),
    ("new, red until fixed (6, 9)", 2, "FAIL_TO_PASS"),
    ("other tests in the file",     5, "PASS_TO_PASS"),
], columns=["group", "n_tests", "list"]))

print(f"\nFAIL_TO_PASS {len(fail_to_pass)}   PASS_TO_PASS {len(pass_to_pass)}")

astropy/modeling/tests/test_separable.py::test_coord_matrix
astropy/modeling/tests/test_separable.py::test_cdot
astropy/modeling/tests/test_separable.py::test_cstack
astropy/modeling/tests/test_separable.py::test_arith_oper
astropy/modeling/tests/test_separable.py::test_separable[compound_model0-result0]
astropy/modeling/tests/test_separable.py::test_separable[compound_model1-result1]
astropy/modeling/tests/test_separable.py::test_separable[compound_model2-result2]
astropy/modeling/tests/test_separable.py::test_separable[compound_model3-result3]
astropy/modeling/tests/test_separable.py::test_separable[compound_model4-result4]
astropy/modeling/tests/test_separable.py::test_separable[compound_model5-result5]
astropy/modeling/tests/test_separable.py::test_separable[compound_model7-result7]
astropy/modeling/tests/test_separable.py::test_separable[compound_model8-result8]
astropy/modeling/tests/test_separable.py::test_custom_model_separable


,group,n_tests,list
0,"existing cases, index 0-5",6,PASS_TO_PASS
1,"new, already green (7, 8)",2,PASS_TO_PASS
2,"new, red until fixed (6, 9)",2,FAIL_TO_PASS
3,other tests in the file,5,PASS_TO_PASS



FAIL_TO_PASS 2   PASS_TO_PASS 13


## 1.5 What the fields add up to

An issue, a fix, a test patch, two lists of names — and no repository. None of
it can be executed. Turning any of it into a number means cloning
`astropy/astropy` at `base_commit`, applying `test_patch`, applying the
candidate patch, and running both lists.

Those four steps *are* the reward function, and all four need a real
environment. §2–3 fake it so this notebook runs offline. §4 is where the faking
stops working.

---
# 2. The mock repository

Real evaluation clones a 2,000-file repo at `base_commit`. We hardcode one
function.

Below is the actual `_cstack` from `separable.py` at that commit — the function
the issue is about. Its two branches are nearly symmetric, and the asymmetry is
the bug:

```
cleft[...]  = left      # uses the operand
cright[...] = 1         # ignores it
```

Everything §3's shell commands return is read out of this one string.

In [24]:
TARGET = "astropy/modeling/separable.py"

# Verbatim from astropy@d16bfe05, trimmed to the one function that matters.
MOCK_FILE = '''def _cstack(left, right):
    """
    Function corresponding to '&' operation.

    Parameters
    ----------
    left, right : `astropy.modeling.Model` or ndarray
        If input is of an array, it is the output of `coord_matrix`.

    Returns
    -------
    result : ndarray
        Result from this operation.

    """
    noutp = _compute_n_outputs(left, right)

    if isinstance(left, Model):
        cleft = _coord_matrix(left, 'left', noutp)
    else:
        cleft = np.zeros((noutp, left.shape[1]))
        cleft[: left.shape[0], : left.shape[1]] = left
    if isinstance(right, Model):
        cright = _coord_matrix(right, 'right', noutp)
    else:
        cright = np.zeros((noutp, right.shape[1]))
        cright[-right.shape[0]:, -right.shape[1]:] = 1

    return np.hstack([cleft, cright])'''

print(f"{TARGET}  ({len(MOCK_FILE.splitlines())} lines)\n")
print(MOCK_FILE)

astropy/modeling/separable.py  (29 lines)

def _cstack(left, right):
    """
    Function corresponding to '&' operation.

    Parameters
    ----------
    left, right : `astropy.modeling.Model` or ndarray
        If input is of an array, it is the output of `coord_matrix`.

    Returns
    -------
    result : ndarray
        Result from this operation.

    """
    noutp = _compute_n_outputs(left, right)

    if isinstance(left, Model):
        cleft = _coord_matrix(left, 'left', noutp)
    else:
        cleft = np.zeros((noutp, left.shape[1]))
        cleft[: left.shape[0], : left.shape[1]] = left
    if isinstance(right, Model):
        cright = _coord_matrix(right, 'right', noutp)
    else:
        cright = np.zeros((noutp, right.shape[1]))
        cright[-right.shape[0]:, -right.shape[1]:] = 1

    return np.hstack([cleft, cright])


---
# 3. A minimal harness with mocked tools

Shaped after mini-swe-agent: **bash only**, no tool-calling API, a strictly
linear message history, one command per turn. The difference is that
`subprocess.run` is replaced by a lookup table.

The mock handles `ls`, `cat`, `grep`, `pytest`, and a heredoc write. Anything
else returns a stub. That's the point — you can see exactly how thin the
environment is.

In [25]:
SYSTEM = f'''You are a software engineering agent. You are in a Python repository.
Fix the bug described in the issue.

Respond with exactly ONE bash command per message, in a fenced block:

```bash
your command here
```

Useful commands:
  ls
  cat {TARGET}
  grep -n "pattern" {TARGET}
  python -m pytest

To rewrite a file:
```bash
cat > {TARGET} <<'EOF'
...full new contents...
EOF
```

One short sentence of reasoning, then exactly one command block.'''

RE_BASH = re.compile(r"```(?:bash|sh)?\s*\n(.*?)```", re.S)
RE_HEREDOC = re.compile(r"cat\s*>\s*(\S+)\s*<<\s*'?EOF'?\n(.*?)\nEOF", re.S)


class MockEnv:
    '''Every method here is a lie. The docstrings say which kind.'''

    def __init__(self):
        self.fs = {TARGET: MOCK_FILE}
        self.orig = dict(self.fs)
        self.calls = []

    def run(self, cmd):
        self.calls.append(cmd)

        m = RE_HEREDOC.search(cmd)
        if m:                                   # real effect on the virtual FS
            path, body = m.group(1), m.group(2)
            self.fs[path] = body
            return f"[mock] wrote {len(body.splitlines())} lines to {path}"

        if cmd.strip().startswith("ls"):        # canned
            return "\n".join(sorted(self.fs)) + "\nsetup.py\nREADME.rst\ntests/"

        if cmd.strip().startswith("cat "):      # real source, one function of it
            path = cmd.split()[-1]
            if path in self.fs:
                return (f"[mock: only the region near the bug is available]\n"
                        + self.fs[path])
            return f"cat: {path}: No such file or directory"

        if cmd.strip().startswith("grep"):      # real search over the fragment
            pat = re.findall(r'"([^"]*)"|\'([^\']*)\'', cmd)
            pat = next((a or b for a, b in pat), "")
            src = self.fs.get(cmd.split()[-1], "")
            hits = [f"{i+1}:{l}" for i, l in enumerate(src.split("\n"))
                    if pat and pat in l]
            return "\n".join(hits) if hits else "(no matches)"

        if "pytest" in cmd:                     # canned failure, real test names
            body = "\n".join(f"FAILED {n}" for n in fail_to_pass)
            return f"[mock] {body}\n=== {len(fail_to_pass)} failed ===\n(mock never actually runs anything)"

        return f"[mock] '{cmd.split()[0]}' not implemented in this stub"

    def patch(self):
        '''The candidate patch: a real unified diff of the virtual FS.'''
        out = []
        for p in self.fs:
            if self.fs[p] != self.orig[p]:
                out += list(difflib.unified_diff(
                    self.orig[p].split("\n"), self.fs[p].split("\n"),
                    fromfile=f"a/{p}", tofile=f"b/{p}", lineterm=""))
        return "\n".join(out)


env = MockEnv()
print(env.run("ls"))
print()
print(env.run(f'grep -n "cright" {TARGET}'))
print()
print(env.run("python -m pytest"))

astropy/modeling/separable.py
setup.py
README.rst
tests/

24:        cright = _coord_matrix(right, 'right', noutp)
26:        cright = np.zeros((noutp, right.shape[1]))
27:        cright[-right.shape[0]:, -right.shape[1]:] = 1
29:    return np.hstack([cleft, cright])

[mock] FAILED astropy/modeling/tests/test_separable.py::test_separable[compound_model6-result6]
FAILED astropy/modeling/tests/test_separable.py::test_separable[compound_model9-result9]
=== 2 failed ===
(mock never actually runs anything)


In [26]:
def rollout(model, tok, max_turns=4, max_new=200, temperature=1.0):
    '''mini-swe-agent shape: linear history, one bash command per turn.'''
    env = MockEnv()
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"ISSUE:\n{inst['problem_statement'][:1500]}"}]

    for _ in range(max_turns):
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ip = tok(prompt, return_tensors="pt", truncation=True, max_length=3072).to(DEV)
        with torch.no_grad():
            out = model.generate(**ip, max_new_tokens=max_new,
                                 do_sample=temperature > 0,
                                 temperature=max(temperature, 1e-5), top_p=0.95,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)
        reply = tok.decode(out[0][ip["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        msgs.append({"role": "assistant", "content": reply})

        m = RE_BASH.search(reply)
        obs = env.run(m.group(1).strip()) if m else \
              "No command found. Reply with exactly one ```bash block."
        msgs.append({"role": "user", "content": obs[:800]})

    return dict(messages=msgs, patch=env.patch(), final=dict(env.fs), calls=env.calls)

---
# 4. Reward

Here is where mocking runs out.

**The real reward** clones the repo at `base_commit`, applies `test_patch`,
applies the candidate patch, runs `FAIL_TO_PASS` and `PASS_TO_PASS`, and returns
`1.0` only if every test in both lists passes. Binary, no partial credit.

We cannot do that offline, and there is no honest shortcut — whether a patch is
correct is not a property of its text. So we don't pretend: the reward below is
**random numbers**. It says nothing about the rollouts. It exists only so §6 has
something to consume.

Every other mock in this notebook costs fidelity. This one removes the signal
entirely, which is exactly the point §7 makes.

In [27]:
def reward_random(n, seed=0):
    '''MOCK reward: a stand-in for a real verifier (unit tests / reward model).
    One value per rollout, seeded for reproducibility. Carries NO information
    about rollout quality -- it only lets us show how the update consumes it.'''
    return np.random.default_rng(seed).random(n).round(3)

print(reward_random(6))

[0.637 0.27  0.041 0.017 0.813 0.913]


---
# 5. Sample a group

In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16 if DEV == "cuda" else torch.float32,
    attn_implementation="sdpa").to(DEV)

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]))
model.print_trainable_parameters()

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3007.37it/s]


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [29]:
G = 6
group = [rollout(model, tok) for _ in range(G)]
reward = reward_random(len(group))

display(pd.DataFrame([
    {
        "rollout":     i,
        "reward":      reward[i],
        "patch_lines": len(r["patch"].split("\n")) if r["patch"] else 0,
        "n_commands":  len(r["calls"]),
    }
    for i, r in enumerate(group)
]))

# full, verbatim trace of every rollout -- commands and candidate patch, nothing hidden
for i, r in enumerate(group):
    print(f"\n================ rollout {i} ================")
    print("commands:")
    for cmd in r["calls"]:
        print("  $", cmd)
    print("\ncandidate patch:")
    print(r["patch"] if r["patch"] else "(no file changed)")

,rollout,reward,patch_lines,n_commands
0,0,0.637,0,4
1,1,0.270,0,1
2,2,0.041,0,3
3,3,0.017,0,4
4,4,0.813,0,3
5,5,0.913,47,3



================ rollout 0 ================
commands:
  $ ls | grep -n 'astropy.modeling.separable.py' | awk '{print $1}'
  $ python setup.py install
  $ mock -m unittest.mock -v 3
  $ mock -m unittest.mock -v 3

candidate patch:
(no file changed)

================ rollout 1 ================
commands:
  $ # Assuming 'Linear1D' and 'Pix2Sky_TAN' are defined earlier in your script
# Convert Linear1D instances to a list of models
linear_models = list(Linear1D(x * 0.1) for x in range(10))

# Compute the separability matrix for each list separately
for i, model in enumerate(linear_models):
    matrix = separability_matrix(model)
    print(f"Separability Matrix for model {i+1}: {matrix}")

# Now the inputs and outputs are separable

candidate patch:
(no file changed)

================ rollout 2 ================
commands:
  $ ls
  $ ls tests/README.rst
  $ ls tests

candidate patch:
(no file changed)

================ rollout 3 ================
commands:
  $ # Use 'grep' to find the line tha

**Degenerate groups.** GRPO's baseline is the group itself, so a group whose
rollouts all score the same has zero spread, zero advantage, and contributes
zero gradient.

That is the normal outcome with a real binary reward: a 0.5B model solves none
of the six, every reward is `0.0`, and the step is a no-op. Keeping groups
non-degenerate — task difficulty, sampling temperature, group size — is most of
the practical work in GRPO. Our random reward sidesteps it by construction.

In [30]:
flat = np.full(len(group), 0.5)

display(pd.DataFrame({
    "group":        ["reward (this run)", "if all rollouts tied"],
    "values":       [str(reward), str(flat)],
    "std":          [reward.std().round(4), flat.std().round(4)],
    "any gradient": [reward.std() > 1e-6, flat.std() > 1e-6],
}))

,group,values,std,any gradient
0,reward (this run),[0.637 0.27 0.041 0.017 0.813 0.913],0.3578,True
1,if all rollouts tied,[0.5 0.5 0.5 0.5 0.5 0.5],0.0000,False


---
# 6. One GRPO step

$$A_i = \frac{r_i - \mathrm{mean}(r)}{\mathrm{std}(r) + \varepsilon}$$

No critic. **The other rollouts are the baseline** — that is the entire idea.
Loss is a policy gradient over assistant tokens only; observation tokens came
from the environment, so crediting them is meaningless.

In [31]:
def build_masked(messages, tokenizer, max_len=3072):
    ids, labels, prev = [], [], ""
    for i, m in enumerate(messages):
        cur = tokenizer.apply_chat_template(messages[:i+1], tokenize=False)
        assert cur.startswith(prev), "chat template is not append-only"
        seg = tokenizer(cur[len(prev):], add_special_tokens=False)["input_ids"]
        ids += seg
        labels += seg if m["role"] == "assistant" else [-100]*len(seg)
        prev = cur
    return ids[:max_len], labels[:max_len]


def seq_logprob(messages):
    ids, labs = build_masked(messages, tok)
    t = torch.tensor([ids], device=DEV)
    msk = torch.tensor([[0. if l == -100 else 1. for l in labs]], device=DEV)[:, 1:]
    logits = model(t).logits[:, :-1]
    lp = torch.log_softmax(logits.float(), -1).gather(-1, t[:, 1:].unsqueeze(-1)).squeeze(-1)
    return (lp * msk).sum() / msk.sum().clamp(min=1), msk.sum().item()

In [32]:
rewards = torch.tensor(reward, dtype=torch.float)
adv = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-4)

print(f"mean reward {rewards.mean():.3f}   std {rewards.std(unbiased=False):.3f}")
display(pd.DataFrame({
    "rollout":    list(range(len(group))),
    "reward":     rewards.numpy().round(3),
    "advantage":  adv.numpy().round(3),
    "sup_tokens": [int(seq_logprob(g["messages"])[1]) for g in group],
}))

mean reward 0.448   std 0.358


,rollout,reward,advantage,sup_tokens
0,0,0.637,0.527,82
1,1,0.270,-0.499,426
2,2,0.041,-1.138,253
3,3,0.017,-1.206,135
4,4,0.813,1.018,385
5,5,0.913,1.298,560


In [33]:
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)

logp_before = [seq_logprob(g["messages"])[0].item() for g in group]

model.train(); opt.zero_grad(set_to_none=True)
loss_total = 0.0
for g, a in zip(group, adv):
    lp, _ = seq_logprob(g["messages"])
    loss = -(a.to(DEV) * lp) / len(group)      # REINFORCE with a group baseline
    loss.backward()
    loss_total += loss.item()

grad_norm = torch.nn.utils.clip_grad_norm_(
    [p for p in model.parameters() if p.requires_grad], 1.0)
opt.step()

logp_after = [seq_logprob(g["messages"])[0].item() for g in group]

print(f"loss {loss_total:+.5f}   grad_norm {grad_norm:.4f}")
display(pd.DataFrame({
    "rollout":     list(range(len(group))),
    "advantage":   [round(a.item(), 3) for a in adv],
    "logp_before": [round(b, 4) for b in logp_before],
    "logp_after":  [round(c, 4) for c in logp_after],
    "delta":       [round(c - b, 5) for b, c in zip(logp_before, logp_after)],
}))

loss -0.07287   grad_norm 0.5630


,rollout,advantage,logp_before,logp_after,delta
0,0,0.527,-2.3340,-2.3369,-0.00297
1,1,-0.499,-1.1735,-1.1736,-0.00003
2,2,-1.138,-1.1900,-1.1905,-0.00057
3,3,-1.206,-1.7507,-1.7553,-0.00458
4,4,1.018,-1.4771,-1.4774,-0.00028
5,5,1.298,-0.6781,-0.6781,0.00001


**Read the last column.** Rollouts with positive advantage should have gained
log-probability; negative-advantage rollouts should have lost it. That is the
entire mechanism — no value network, no reward model, just *this trajectory
scored better than its siblings, so make it more likely.*

One step. Effect size is tiny, as it should be at lr=1e-5.

---
# 7. The honest ledger

What we faked, and what each shortcut cost:

| Faked | Real version | What the fake hides |
|---|---|---|
| repo → one hardcoded function | full clone at `base_commit` | the agent can't explore, so localization — most of the real difficulty — vanishes |
| `cat`/`grep` → dict lookup | shell in a container | no build, no imports, no cross-file reasoning |
| `pytest` → canned string | real suite at real commit | **everything**; see below |
| reward → random numbers | `FAIL_TO_PASS ∧ PASS_TO_PASS` | there is no learning signal at all |
| 1 task | 500 (Verified) / 50k (SWE-smith) | no generalization claim is possible |
| 1 step | thousands | no learning |

The first three cost **fidelity** — a worse environment, but still an
environment. The fourth is different in kind. Sections 5 and 6 ran real
arithmetic on meaningless numbers: real rollouts, a real advantage
calculation, a real gradient, and a real optimizer step, all driven by
`np.random`. Every mechanism worked; the loop learned nothing, and could not.

This is why production SWE RL spends its budget on container fleets, and why
the verifier — not the GPU — is usually the bottleneck.

Good closing question for the room: *every number in §6 was computed correctly,
and the run was still worthless. What is the smallest change that would fix
that?*

---
## Where to go next

- **Lab 1** — trajectory SFT, loss masking, real SWE-smith data
- **Lab 2** — a full GRPO loop with real execution rewards, on a shrunken gym
- Real frameworks: `NovaSky-AI/SkyRL`, `PrimeIntellect-ai/prime-rl`
- The harness this imitates: `SWE-agent/mini-swe-agent` (pin `<2`)